# Marco 4: modelos M1 em sessão única (T-4.3b, D-53, D-64)

Notebook sem lógica: monta o Drive, carrega o config, confere o ambiente e dispara a sessão em segundo plano (`python -m dengue_sinan.runner`). O trabalho todo fica em `src/dengue_sinan/runner.py`.

Pasta do projeto no Drive: `MyDrive/dengue-comorbidity-risk/`. Ajuste `PASTA_DRIVE` se for outra.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Código no GitHub, commitado pelo orquestrador antes desta execução (D-66).
import os

REPO_URL = 'https://github.com/PabloThePmX/dengue-comorbidity-risk'
PASTA_REPO = '/content/dengue-comorbidity-risk'

if not os.path.isdir(PASTA_REPO):
    !git clone -q {REPO_URL} {PASTA_REPO}
%cd {PASTA_REPO}
!git pull -q
!pip install -q -e .


In [ ]:
from dengue_sinan import runner
from dengue_sinan.config import load_config

PASTA_DRIVE = '/content/drive/MyDrive/dengue-comorbidity-risk'

cfg = load_config()  # config/base.yaml do clone
config_sessao = runner.config_da_sessao(cfg, PASTA_DRIVE)  # dados e mlruns no Drive
LOG = f'{PASTA_DRIVE}/logs/sessao-m1.log'
print('config da sessão:', config_sessao)


In [ ]:
# TabFM (D-69): instalado só nesta sessão do Colab, fora do lock; a versão vem do config
# (tabfm.versao_pacote). Esta célula só instala; não há lógica de treino aqui.
versao_tabfm = cfg['tabfm']['versao_pacote']
!pip install -q "tabfm[pytorch]=={versao_tabfm}"


In [ ]:
# Confere GPU e RAM contra cfg.ambiente antes de começar (D-64).
from dengue_sinan import verify

for check in verify.environment(load_config(config_sessao)):
    print(check)

In [ ]:
# Segundo plano: a sessão continua mesmo se esta célula terminar. Log no Drive.
!mkdir -p {PASTA_DRIVE}/logs
!nohup python -m dengue_sinan.runner --config {config_sessao} > {LOG} 2>&1 &
print('sessão iniciada; acompanhe com a célula seguinte')

In [ ]:
# Célula curta de acompanhamento: lê o relatório da sessão e o fim do log.
import json
from pathlib import Path

relatorio = runner.caminho_relatorio(load_config(config_sessao))
if relatorio.exists():
    dados = json.loads(relatorio.read_text(encoding='utf-8'))
    for bloco in dados['blocos']:
        print(bloco['bloco'], 'passou =', bloco['passou'])
    print('sessao_completa =', dados['sessao_completa'], '| passou =', dados['passou'])
else:
    print('relatório ainda não existe')
print(Path(LOG).read_text(encoding='utf-8')[-1500:])